# Redis Pub/Sub

## Overview
Redis Pub/Sub is a **fire-and-forget messaging system** built into Redis. Publishers send messages to named channels; subscribers receive them in real time. No message history, no delivery guarantees — but zero overhead and microsecond fanout.

**Pub/Sub vs Redis Streams:**

| Feature | Pub/Sub | Redis Streams |
|---|---|---|
| Message persistence | None | Full log retained |
| Delivery guarantee | Fire-and-forget | At-least-once (consumer groups) |
| Consumer groups | No — all receive all | Yes — work distributed |
| Replay | No | Yes (from any offset) |
| Use case | Real-time notifications | Event sourcing, task queues |

**Core commands:**
```python
pubsub = r.pubsub()
pubsub.subscribe('channel')                    # exact channel
pubsub.psubscribe('alerts:*')                  # glob pattern
r.publish('channel', json.dumps(payload))      # send a message
for msg in pubsub.listen(): ...                # blocking receive loop
thread = pubsub.run_in_thread(sleep_time=0.01) # non-blocking background thread
```

---

In [ ]:
import fakeredis, json, time

# fakeredis: full redis-py API, no server required
# Replace with redis.Redis(host='...') for a live server
r = fakeredis.FakeRedis(decode_responses=True)
print('Redis ready:', r.ping())


In [ ]:
import fakeredis, threading, time, json

r  = fakeredis.FakeRedis(decode_responses=True)
r2 = fakeredis.FakeRedis(decode_responses=True)  # separate connection for subscriber

print("=== Pub/Sub: publish-subscribe messaging ===")
print()
print("Redis Pub/Sub key concepts:")
concepts = [
    ("Publisher",    "Sends messages to a named channel — does not know who is listening"),
    ("Subscriber",   "Listens on one or more channels — does not know who is publishing"),
    ("Channel",      "Named message bus — no persistence; messages fire and forget"),
    ("Pattern",      "PSUBSCRIBE 'alerts:*' matches alerts:critical, alerts:info, etc."),
    ("Fire-forget",  "No message acknowledgement; no delivery guarantee; not a queue"),
    ("No history",   "Subscribers only receive messages published AFTER they subscribe"),
]
for name, desc in concepts:
    print(f"  {name:<14s}  {desc}")

print()
print("Pub/Sub vs Redis Streams (choose based on need):")
comparison = [
    ("Delivery guarantee", "None — fire-and-forget",       "At-least-once — consumer groups"),
    ("Message history",    "None — past messages lost",     "Full log retained"),
    ("Consumer groups",    "No — all subscribers get all",  "Yes — work distributed"),
    ("Replay",             "No",                            "Yes (from any offset)"),
    ("Use case",           "Real-time notifications",       "Event sourcing, task queues"),
]
print(f"  {'Feature':<22s}  {'Pub/Sub':<36s}  Redis Streams")
print("  " + "-"*76)
for feat, ps, stream in comparison:
    print(f"  {feat:<22s}  {ps:<36s}  {stream}")


---
## Subscribe and receive messages

In [ ]:
import threading, time, queue

received_msgs = queue.Queue()

def subscriber_thread():
    """Subscribe and collect messages — runs in background thread."""
    sub_r = fakeredis.FakeRedis(decode_responses=True)
    pubsub = sub_r.pubsub()
    pubsub.subscribe('alerts:lab', 'alerts:medication', 'notifications:all')
    print("  Subscriber: listening on alerts:lab, alerts:medication, notifications:all")
    timeout = time.time() + 2.0
    for message in pubsub.listen():
        if message['type'] == 'message':
            received_msgs.put(message)
        if time.time() > timeout:
            break
    pubsub.unsubscribe()
    sub_r.close()

print("=== Subscribe and receive messages ===")
print()

# Start subscriber in background
t = threading.Thread(target=subscriber_thread, daemon=True)
t.start()
time.sleep(0.1)  # let subscriber connect

# Publish messages
pub_r = fakeredis.FakeRedis(decode_responses=True)
messages = [
    ('alerts:lab',        {'patient': 'P003', 'test': 'HbA1c', 'value': 9.1, 'severity': 'high'}),
    ('alerts:medication', {'patient': 'P005', 'drug': 'Insulin glargine', 'action': 'refill_needed'}),
    ('notifications:all', {'type': 'system', 'msg': 'Scheduled maintenance 02:00 UTC'}),
    ('alerts:lab',        {'patient': 'P001', 'test': 'eGFR', 'value': 45, 'severity': 'moderate'}),
]
print("Publisher sending messages:")
for channel, payload in messages:
    n_receivers = pub_r.publish(channel, json.dumps(payload))
    print(f"  PUBLISH {channel:<24s}  {n_receivers} receiver(s)  payload={payload}")
    time.sleep(0.05)

t.join(timeout=2.5)

print()
print("Subscriber received:")
while not received_msgs.empty():
    msg = received_msgs.get()
    data = json.loads(msg['data'])
    print(f"  channel={msg['channel']:<24s}  data={data}")

pub_r.close()


---
## Pattern subscribe and message handlers

In [ ]:
print("=== Pattern subscribe (PSUBSCRIBE) ===")
print()

pattern_code = """
# PSUBSCRIBE: subscribe using glob-style patterns
pubsub = r.pubsub()
pubsub.psubscribe('alerts:*')         # matches alerts:lab, alerts:medication, alerts:critical
pubsub.psubscribe('patient:P001:*')   # all events for a specific patient

for message in pubsub.listen():
    if message['type'] == 'pmessage':   # pmessage for pattern, message for exact channel
        pattern = message['pattern']    # e.g., 'alerts:*'
        channel = message['channel']    # actual channel: 'alerts:lab'
        data    = json.loads(message['data'])
        print(f"Pattern {pattern}: {channel} -> {data}")
"""
print("Pattern subscribe code:")
print(pattern_code)

print("=== Message handler callbacks ===")
handler_code = """
# Register per-channel handlers (non-blocking with run_in_thread)
def on_lab_alert(message):
    data = json.loads(message['data'])
    if float(data.get('value', 0)) > 8.0:
        send_alert_email(data['patient'], data['test'], data['value'])

def on_medication_alert(message):
    data = json.loads(message['data'])
    trigger_refill_workflow(data['patient'], data['drug'])

pubsub = r.pubsub()
pubsub.subscribe(**{
    'alerts:lab':        on_lab_alert,
    'alerts:medication': on_medication_alert,
})

# Run subscriber in background thread (non-blocking):
thread = pubsub.run_in_thread(sleep_time=0.01, daemon=True)
# thread.stop() to shut down cleanly
"""
print("Handler callbacks:")
print(handler_code)

print("Production pub/sub patterns:")
patterns = [
    ("Real-time clinical alerts",    "Lab result published -> alert service subscriber"),
    ("Cache invalidation broadcast", "DB update -> PUBLISH invalidate:patient:P001"),
    ("Live dashboard updates",       "WebSocket server subscribes; publishes to browser"),
    ("Microservice events",          "Order service publishes; email/SMS services subscribe"),
    ("Chat / notifications",         "User-specific channels: user:P001:notifications"),
]
for name, desc in patterns:
    print(f"  {name:<32s}  {desc}")


---
## Common Pitfalls

**1. Using Pub/Sub when you need guaranteed delivery**
Redis Pub/Sub is fire-and-forget. If a subscriber is offline, it misses all messages published while it was down. For guaranteed delivery, use Redis Streams (XADD/XREAD) or a dedicated message broker (RabbitMQ, Kafka).

**2. Subscribing on a shared connection used for other commands**
Once a connection enters subscribe mode, it can only issue SUBSCRIBE, UNSUBSCRIBE, PSUBSCRIBE, PUNSUBSCRIBE, PING, and QUIT. Any other command raises `redis.exceptions.ResponseError`. Always use a dedicated connection for the subscriber.

**3. Not handling `subscribe` and `unsubscribe` message types**
`pubsub.listen()` yields messages of type `'subscribe'`, `'unsubscribe'`, and `'message'`. Processing only `'message'` type is correct. The other types are control messages, not data — parsing them as JSON raises an error.

**4. Message data not JSON — assuming string structure**
Publishers can send any string. If you expect JSON, always wrap `json.loads()` in a try/except to handle malformed messages without crashing the subscriber.

**5. run_in_thread() without a stop mechanism**
`thread = pubsub.run_in_thread(sleep_time=0.01)` returns a `PubSubWorkerThread`. In tests or short-lived processes, forgetting to call `thread.stop()` leaves a daemon thread running. Always stop the thread in cleanup code.

**6. Subscribing to too many channels with PSUBSCRIBE '*'**
`PSUBSCRIBE '*'` receives every message published on any channel. On a busy Redis instance, this can flood the subscriber with thousands of messages per second. Use specific channel prefixes and only subscribe to what the service actually needs to process.


---
*sql_methods_library - Samantha McGarrigle*